# Arabic IR Pipeline — Main Notebook

**Improving Arabic Information Retrieval and Reranking Performance using Knowledge Distillation**

This notebook orchestrates the full pipeline:
1. **Configure** — fill in the config cell below
2. **W&B setup** — initialise Weights & Biases run (optional)
3. **Load data** — queries, corpus, train/dev splits
4. **Mine hard negatives** — BM25 + ANCE curriculum
5. **Train** — bi-encoder and/or cross-encoder (with/without KD, LoRA, Matryoshka)
6. **Retrieve** — FAISS dense first-stage
7. **Rerank** — cross-encoder second stage
8. **Evaluate** — MRR@10, NDCG@10, Recall@K, MAP@10
9. **Report** — auto-generated HTML report + W&B artifacts

## 0. Install dependencies

In [ ]:
!pip install sentence_transformers transformers datasets huggingface_hub peft faiss-gpu accelerate wandb -q
!pip install matplotlib pandas -q
# Pyserini (BM25) requires Java — uncomment if needed
# !sudo apt-get install openjdk-21-jre -y && pip install pyserini -q

## 1. Configuration

**Edit only this cell** to control the entire pipeline.

Supported `base_model` values:
| Key | HuggingFace ID | Use as |
|-----|---------------|--------|
| `"AraELECTRA"` | `aubmindlab/araelectra-base-discriminator` | bi-encoder / cross-encoder |
| `"AraDPR"` | `abdoelsayed/AraDPR` | bi-encoder / cross-encoder |
| `"mMiniLML"` | `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` | bi-encoder |
| `"mMiniLMv2CE"` | `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1` | cross-encoder / KD teacher |

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

from src.config.config import (
    PipelineConfig, DataConfig, ModelConfig,
    TrainingConfig, EvalConfig, LoRAConfig
)

config = PipelineConfig(
    # ── Dataset ──────────────────────────────────────────────────────────
    data=DataConfig(
        dataset="mmarco",          # "mmarco" | "mrtydi"
        max_train_samples=None,    # None = full dataset; set e.g. 100_000 to test quickly
    ),

    # ── Model ────────────────────────────────────────────────────────────
    model=ModelConfig(
        base_model="mMiniLMv2CE",  # "AraELECTRA" | "AraDPR" | "mMiniLML" | "mMiniLMv2CE"
        encoder_type="cross",      # "bi" | "cross" | "both"
        max_length=256,

        # LoRA — set use_lora=True to enable
        use_lora=False,
        lora_config=LoRAConfig(
            r=16,
            lora_alpha=32,
            target_modules=None,   # None = auto-detect per architecture
            lora_dropout=0.1,
        ),

        # Matryoshka (bi-encoder only)
        use_matryoshka=False,
        matryoshka_dims=[768, 512, 256, 128],
    ),

    # ── Training ─────────────────────────────────────────────────────────
    training=TrainingConfig(
        # Knowledge distillation
        use_kd=True,
        kd_mode="listwise",        # "pairwise" | "listwise"  (cross-encoder)
        kd_alpha=0.5,              # α·BCE_hard + (1-α)·KL_soft
        kd_temperature=1.0,

        # Hard negatives
        hard_negatives=True,
        hn_strategy="curriculum",  # "bm25" | "ance" | "combined" | "curriculum"
        hn_bm25_top_k=200,
        hn_ance_top_k=200,
        hn_refresh_steps=5000,
        curriculum_schedule=[(1, "bm25"), (2, "combined"), (3, "ance")],

        # Hybrid bi-encoder loss (MNRL + MarginMSE)
        use_mnrl_hybrid=True,
        mnrl_weight=0.5,

        # Standard hyperparameters
        num_train_epochs=3,
        per_device_train_batch_size=16,
        gradient_accumulation_steps=8,
        learning_rate=7e-5,
        weight_decay=0.0,
        warmup_ratio=0.07,
        fp16=True,
        eval_steps=2000,
        output_dir="./training_output",

        # W&B — set report_to="wandb" to enable
        report_to=None,            # None | "wandb" | "tensorboard"
        run_name="arabic-ir-kd",
        wandb_project="arabic-ir-kd",
        wandb_entity=None,         # your W&B username or team, or None
        wandb_api_key=None,        # or set WANDB_API_KEY env var
        wandb_log_artifacts=True,  # upload HTML report + checkpoints as artifacts
    ),

    # ── Evaluation ───────────────────────────────────────────────────────
    evaluation=EvalConfig(
        metrics=["mrr@10", "ndcg@10", "recall@10", "recall@100", "recall@1000", "map@10"],
        eval_batch_size=64,
        first_stage_top_k=1000,
        rerank_top_k=1000,
    ),

    # ── Report & Hub ─────────────────────────────────────────────────────
    report_output="./report.html",
    push_to_hub=False,
    hub_token=None,
    hub_repo_id=None,
    seed=42,
)

print(config.summary())

## 2. Weights & Biases Setup

Set `report_to="wandb"` in the config above to enable logging.
This cell initialises the run and is a no-op when W&B is disabled.

In [ ]:
from src.utils import wandb_logger

wb_run = wandb_logger.init(config)

if wb_run:
    print(f"W&B run active: {wb_run.url}")
    print(f"  Project : {wb_run.project}")
    print(f"  Run name: {wb_run.name}")
else:
    print("W&B disabled (set report_to='wandb' to enable)")

## 3. Load Data

In [ ]:
from src.data.loader import DatasetLoader

loader = DatasetLoader(config.data)

print("Loading queries…")
queries = loader.load_queries(split="dev")
print(f"  {len(queries)} queries")

print("Loading corpus…")
corpus = loader.load_corpus()
print(f"  {len(corpus)} documents")

print("Loading dev samples…")
dev_samples = loader.load_dev_samples()
print(f"  {len(dev_samples)} dev queries")

print("Loading qrels…")
qrels = loader.load_qrels()
print(f"  {len(qrels)} qrels entries")

In [ ]:
# Load training dataset (only needed for training, skip if evaluating only)
train_dataset = loader.load_train_dataset()
print(train_dataset)

## 4. Hard Negative Mining

In [ ]:
from src.data.hard_negatives import HardNegativeMiner
from src.retrieval.bm25_retrieval import BM25Retriever

miner = HardNegativeMiner(
    strategy=config.training.hn_strategy,
    bm25_top_k=config.training.hn_bm25_top_k,
    ance_top_k=config.training.hn_ance_top_k,
    curriculum_schedule=config.training.curriculum_schedule,
    seed=config.seed,
)

# Store queries/corpus references for ANCE refresh callback during training
miner._queries = queries
miner._corpus = corpus

if config.training.hard_negatives:
    bm25 = BM25Retriever()
    bm25_run_path = os.path.join(config.data.cache_dir, "bm25_run.txt")
    if os.path.exists(bm25_run_path):
        print("Loading cached BM25 run…")
        bm25_run = bm25.load_run(bm25_run_path, k=config.training.hn_bm25_top_k)
        miner.set_bm25_negatives(bm25_run, qrels)
        print(f"  BM25 hard negatives ready for {len(miner._bm25_negatives)} queries")
    else:
        print("No cached BM25 run found — hard_negatives will fall back to ANCE only.")
        print("Use BM25Retriever.retrieve() with Pyserini to generate one.")

## 5. Train Bi-Encoder

In [ ]:
from src.models.bi_encoder import BiEncoderModel
from src.training.bi_encoder_trainer import BiEncoderTrainer

bi_model = None

if config.model.encoder_type in ("bi", "both"):
    print("=== Training Bi-Encoder ===")
    bi_model = BiEncoderModel(config.model)
    bi_trainer = BiEncoderTrainer(config, bi_model)

    bi_trainer_obj = bi_trainer.train(
        train_dataset=train_dataset,
        dev_samples=dev_samples,
        hard_negative_miner=miner if config.training.hard_negatives else None,
    )

    bi_save_path = os.path.join(config.training.output_dir, "bi_encoder", "final")
    bi_model.save(bi_save_path)
    print(f"Bi-encoder saved → {bi_save_path}")

    if config.training.wandb_log_artifacts and config.use_wandb:
        wandb_logger.log_artifact(bi_save_path, "bi-encoder-model", "model")

    if config.push_to_hub and config.hub_repo_id:
        bi_model.push_to_hub(config.hub_repo_id + "-bi", token=config.hub_token)
else:
    print("Skipping bi-encoder training")

## 6. Train Cross-Encoder

> `mMiniLMv2CE` (`cross-encoder/mmarco-mMiniLMv2-L12-H384-v1`) is a strong multilingual
> cross-encoder pre-trained on mMARCO. Fine-tuning it on Arabic with KD gives a powerful reranker.

In [ ]:
from src.models.cross_encoder import CrossEncoderModel
from src.training.cross_encoder_trainer import CrossEncoderTrainer

ce_model = None

if config.model.encoder_type in ("cross", "both"):
    print("=== Training Cross-Encoder ===")
    ce_model = CrossEncoderModel(config.model)
    ce_trainer = CrossEncoderTrainer(config, ce_model)

    ce_trainer_obj = ce_trainer.train(
        train_dataset=train_dataset,
        dev_samples=dev_samples,
    )

    ce_save_path = os.path.join(config.training.output_dir, "cross_encoder", "final")
    ce_model.save(ce_save_path)
    print(f"Cross-encoder saved → {ce_save_path}")

    if config.training.wandb_log_artifacts and config.use_wandb:
        wandb_logger.log_artifact(ce_save_path, "cross-encoder-model", "model")

    if config.push_to_hub and config.hub_repo_id:
        ce_model.push_to_hub(config.hub_repo_id + "-ce", token=config.hub_token)
else:
    print("Skipping cross-encoder training")

## 7. First-Stage Retrieval (FAISS)

In [ ]:
from src.retrieval.faiss_retrieval import FaissRetriever

if bi_model is None:
    pretrained_bi = "hatemestinbejaia/mmarco-Arabic-AraDPR-bi-encoder-KD-v1"
    bi_model = BiEncoderModel.from_pretrained(pretrained_bi, config.model)

retriever = FaissRetriever(
    bi_encoder=bi_model.get_sentence_transformer(),
    top_k=config.evaluation.first_stage_top_k,
    batch_size=128,
)

print("Building FAISS index…")
retriever.build_index(corpus)

print("Retrieving…")
first_stage_run = retriever.retrieve(queries)
print(f"Retrieved for {len(first_stage_run)} queries")

run_path = os.path.join(config.training.output_dir, "first_stage_run.tsv")
retriever.save_run(first_stage_run, run_path)

## 8. Reranking

In [ ]:
from sentence_transformers.cross_encoder import CrossEncoder
from src.evaluation.evaluator import PipelineEvaluator

if ce_model is None:
    pretrained_ce = "hatemestinbejaia/mmarco-Arabic-AraDPR-cross-encoder-KD-v1"
    ce_reranker = CrossEncoder(pretrained_ce, max_length=config.model.max_length)
else:
    ce_save_path = os.path.join(config.training.output_dir, "cross_encoder", "final")
    ce_reranker = CrossEncoder(ce_save_path, max_length=config.model.max_length)

evaluator = PipelineEvaluator(config, queries, corpus, qrels)

print("Reranking…")
reranked_run = evaluator._rerank(
    reranker=ce_reranker,
    first_stage_run=first_stage_run,
    top_k=config.evaluation.rerank_top_k,
    use_cross_encoder=True,
)
print("Reranking done.")

## 9. Evaluate

In [ ]:
import pandas as pd

kd_label = config.training.kd_mode if config.training.use_kd else "nokd"
model_label = config.model.base_model

fs_result = evaluator.evaluate_first_stage(
    run=first_stage_run,
    model_name=model_label,
    kd_mode=kd_label,
)

rr_result = evaluator.evaluate_reranking(
    reranker=ce_reranker,
    first_stage_run=first_stage_run,
    model_name=model_label,
    kd_mode=kd_label,
    use_cross_encoder=True,
)

all_results = [fs_result, rr_result]

# Log final metrics to W&B summary and table
wandb_logger.log_results(all_results, config)
wandb_logger.log_summary({
    "final/first_stage_MRR@10":  fs_result.mrr_at_10,
    "final/first_stage_R@1000":  fs_result.recall_at_1000,
    "final/reranking_MRR@10":    rr_result.mrr_at_10,
    "final/reranking_NDCG@10":   rr_result.ndcg_at_10,
})

df = pd.DataFrame([r.as_dict() for r in all_results])
display(df)

## 10. Generate HTML Report

In [ ]:
from src.report.reporter import ReportGenerator

reporter = ReportGenerator(config, results=all_results)

reporter.plot_metrics_bar(output_path="charts/mrr10_bar.png",       metric="MRR@10")
reporter.plot_metrics_bar(output_path="charts/ndcg10_bar.png",      metric="NDCG@10")
reporter.plot_metrics_bar(output_path="charts/recall1000_bar.png",  metric="R@1000")

report_path = reporter.generate()

# Upload HTML report and charts to W&B as an artifact
if config.training.wandb_log_artifacts and config.use_wandb:
    wandb_logger.log_artifact(report_path,  "pipeline-report", "report", "HTML evaluation report")
    wandb_logger.log_artifact("charts/",    "metric-charts",   "report", "Metric bar charts")

from IPython.display import IFrame
IFrame(report_path, width='100%', height=700)

## 11. Finish W&B Run

In [ ]:
wandb_logger.finish()
print("Done.")

## 12. (Optional) Multi-Model Evaluation

Skip sections 5–6 and run this cell to benchmark all pre-trained models at once.
Results and charts are logged to W&B if enabled.

In [ ]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.cross_encoder import CrossEncoder
from src.evaluation.evaluator import PipelineEvaluator
from src.report.reporter import ReportGenerator
from src.utils import wandb_logger

# (bi_encoder_id, cross_encoder_id, label, kd_mode)
model_pairs = [
    ("hatemestinbejaia/mmarco-Arabic-AraDPR-bi-encoder-NoKD-v1",
     "hatemestinbejaia/mmarco-Arabic-AraDPR-cross-encoder-NoKD-v1",
     "AraDPR", "nokd"),
    ("hatemestinbejaia/mmarco-Arabic-AraDPR-bi-encoder-KD-v1",
     "hatemestinbejaia/mmarco-Arabic-AraDPR-cross-encoder-KD-v1",
     "AraDPR", "pairwise_kd"),
    ("hatemestinbejaia/mmarco-Arabic-AraElectra-bi-encoder-KD-v1",
     "hatemestinbejaia/mmarco-Arabic-AraElectra-cross-encoder-KD-v1",
     "AraELECTRA", "pairwise_kd"),
    ("hatemestinbejaia/mmarco-Arabic-mMiniLML-bi-encoder-KD-v1",
     "hatemestinbejaia/mmarco-Arabic-mMiniLML-cross-encoder-KD-v1",
     "mMiniLML", "pairwise_kd"),
    # mMiniLMv2CE used as a zero-shot multilingual reranker baseline
    ("hatemestinbejaia/mmarco-Arabic-mMiniLML-bi-encoder-KD-v1",
     "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
     "mMiniLMv2CE (zero-shot)", "nokd"),
]

wb_run = wandb_logger.init(config)   # no-op if already initialised or disabled

multi_results = []
evaluator = PipelineEvaluator(config, queries, corpus, qrels)

for bi_id, ce_id, label, kd_mode in model_pairs:
    print(f"\n{'='*60}\nEvaluating {label} ({kd_mode})")

    bi = SentenceTransformer(bi_id)
    fs_retriever = FaissRetriever(bi_encoder=bi, top_k=1000)
    fs_retriever.build_index(corpus)
    run = fs_retriever.retrieve(queries)
    fs_res = evaluator.evaluate_first_stage(run, model_name=label, kd_mode=kd_mode)
    print(f"  First-stage: MRR@10={fs_res.mrr_at_10:.4f}  R@1000={fs_res.recall_at_1000:.4f}")

    ce = CrossEncoder(ce_id, max_length=256)
    rr_res = evaluator.evaluate_reranking(ce, run, model_name=label, kd_mode=kd_mode)
    print(f"  Reranking:   MRR@10={rr_res.mrr_at_10:.4f}  NDCG@10={rr_res.ndcg_at_10:.4f}")

    wandb_logger.log_metrics({
        f"{label}/first_stage_MRR@10": fs_res.mrr_at_10,
        f"{label}/first_stage_R@1000": fs_res.recall_at_1000,
        f"{label}/reranking_MRR@10":   rr_res.mrr_at_10,
        f"{label}/reranking_NDCG@10":  rr_res.ndcg_at_10,
    })
    multi_results.extend([fs_res, rr_res])

# Report
reporter = ReportGenerator(config, results=multi_results)
reporter.plot_metrics_bar("charts/all_mrr10.png",      "MRR@10")
reporter.plot_metrics_bar("charts/all_ndcg10.png",     "NDCG@10")
reporter.plot_metrics_bar("charts/all_recall1000.png", "R@1000")
report_path = reporter.generate()

wandb_logger.log_results(multi_results, config)
if config.training.wandb_log_artifacts and config.use_wandb:
    wandb_logger.log_artifact(report_path, "multi-model-report", "report")

wandb_logger.finish()

df = pd.DataFrame([r.as_dict() for r in multi_results])
display(df)